In [2]:
!pip install numpy matplotlib
!pip install scipy
import os
print(os.getcwd())


  Using cached numpy-2.1.3-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (62 kB)
  Using cached matplotlib-3.9.2-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (11 kB)
  Using cached contourpy-1.3.1-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (5.4 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 164.5/164.5 kB 4.9 MB/s eta 0:00:00
  Using cached kiwisolver-1.4.7-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (6.3 kB)
  Using cached pillow-11.0.0-cp311-cp311-manylinux_2_28_x86_64.whl.metadata (9.1 kB)
  Using cached pyparsing-3.2.0-py3-none-any.whl.metadata (5.0 kB)
Using cached numpy-2.1.3-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (16.3 MB)
Using cached matplotlib-3.9.2-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (8.3 MB)
Using cached contourpy-1.3.1-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.

In [4]:
import re
import numpy as np
from scipy.stats import entropy

def read_data(file_path):
    data = {}
    with open(file_path, 'r') as file:
        next(file)
        for line in file:
            if "PASS" not in line:
                continue
            parts = line.strip().split('\t')
            if len(parts) < 5:
                continue
            fragments = parts[4].split(';')
            for frag in fragments:
                match = re.match(r'(chr\w+):(\d+)-(\d+)', frag)
                if match:
                    chrom, start, end = match.groups()
                    if chrom not in data:
                        data[chrom] = []
                    data[chrom].append(frag)
    return data

def convert_to_bin_counts(fragments, bin_size=500):
    bin_counts = {}
    for fragment in fragments:
        chrom, coords = fragment.split(':')
        start, end = map(int, coords.split('-'))
        bin_start = start // bin_size + 1
        bin_end = end // bin_size + 1
        mid_bin = (bin_start + bin_end) // 2
        bin_counts[mid_bin] = bin_counts.get(mid_bin, 0) + 1
    return bin_counts

def compute_distance_counts(bin_counts):
    bins = list(bin_counts.keys())
    counts = list(bin_counts.values())
    distance_counts = {}
    for i in range(len(bins)):
        bin_i = bins[i]
        count_i = counts[i]
        for j in range(i, len(bins)):
            bin_j = bins[j]
            count_j = counts[j]
            distance = abs(bin_i - bin_j)
            if i == j:
                pair_count = count_i * (count_i - 1) // 2
            else:
                pair_count = count_i * count_j
            if pair_count > 0:
                distance_counts[distance] = distance_counts.get(distance, 0) + pair_count
    return distance_counts

def compute_pmf(distance_counts):
    total_pairs = sum(distance_counts.values())
    pmf = {d: count / total_pairs for d, count in distance_counts.items()}
    return pmf

def calculate_symmetric_kl_divergence(pmf1, pmf2):
    all_distances = set(pmf1.keys()).union(set(pmf2.keys()))
    prob1 = np.array([pmf1.get(d, 0) for d in all_distances])
    prob2 = np.array([pmf2.get(d, 0) for d in all_distances])
    valid = (prob1 > 0) & (prob2 > 0)
    if not np.any(valid):
        return np.nan
    prob1_filtered = prob1[valid]
    prob2_filtered = prob2[valid]
    prob1_filtered /= prob1_filtered.sum()
    prob2_filtered /= prob2_filtered.sum()
    kl_div_pq = entropy(prob1_filtered, prob2_filtered)
    kl_div_qp = entropy(prob2_filtered, prob1_filtered)
    symmetric_kl = 0.5 * (kl_div_pq + kl_div_qp)
    return symmetric_kl

def main(file_path):
    data = read_data(file_path)
    chromosome_pmf = {}
    for chrom, fragments in data.items():
        bin_counts = convert_to_bin_counts(fragments)
        if len(bin_counts) < 2:
            continue
        distance_counts = compute_distance_counts(bin_counts)
        pmf = compute_pmf(distance_counts)
        chromosome_pmf[chrom] = pmf
    first_group = ["chr2L", "chr2R", "chr3L", "chr3R"]
    second_group = ["chrX", "chr4"]
    def compute_kl(chr1, chr2):
        if chr1 in chromosome_pmf and chr2 in chromosome_pmf:
            pmf1 = chromosome_pmf[chr1]
            pmf2 = chromosome_pmf[chr2]
            kl_divergence = calculate_symmetric_kl_divergence(pmf1, pmf2)
            print(f"KL divergence between {chr1} and {chr2}: {kl_divergence}")
    print("KL Divergence within group 2L, 2R, 3L, 3R:")
    for i in range(len(first_group)):
        for j in range(i+1, len(first_group)):
            compute_kl(first_group[i], first_group[j])
    print("\nKL Divergence between 2L, 2R, 3L, 3R and X, 4:")
    for chrom_main in first_group:
        for chrom_other in second_group:
            compute_kl(chrom_main, chrom_other)

if __name__ == "__main__":
    file_path = '/home/hzhou53/2024 Fall DNA and GIN model/GSM3347525NR_FDR_0.1_pseudoGEM_10000_enrichTest_master.txt'
    main(file_path)


KL Divergence within group 2L, 2R, 3L, 3R:
KL divergence between chr2L and chr2R: 0.06018911119926895
KL divergence between chr2L and chr3L: 0.0615363534182851
KL divergence between chr2L and chr3R: 0.04201401809695958
KL divergence between chr2R and chr3L: 0.11046316092843478
KL divergence between chr2R and chr3R: 0.10209715827999535
KL divergence between chr3L and chr3R: 0.03099035752939059

KL Divergence between 2L, 2R, 3L, 3R and X, 4:
KL divergence between chr2L and chrX: 0.14885209204789895
KL divergence between chr2L and chr4: 0.2210837212846546
KL divergence between chr2R and chrX: 0.057780127872799694
KL divergence between chr2R and chr4: 0.21572088917568516
KL divergence between chr3L and chrX: 0.19881896157829176
KL divergence between chr3L and chr4: 0.1452654929068528
KL divergence between chr3R and chrX: 0.19144702899919003
KL divergence between chr3R and chr4: 0.26433817554685524


In [3]:
import re
import numpy as np
from scipy.stats import entropy

def read_data(file_path):
    data = {}
    with open(file_path, 'r') as file:
        next(file)
        for line in file:
            if "PASS" not in line:
                continue
            parts = line.strip().split('\t')
            if len(parts) < 5:
                continue
            fragments = parts[4].split(';')
            for frag in fragments:
                match = re.match(r'(chr\w+):(\d+)-(\d+)', frag)
                if match:
                    chrom, start, end = match.groups()
                    if chrom not in data:
                        data[chrom] = []
                    data[chrom].append(int(start))
                    data[chrom].append(int(end))
    return data

def convert_to_bin_counts(fragments, bin_size=500):
    bin_counts = {}
    for i in range(0, len(fragments), 2):
        start = fragments[i]
        end = fragments[i+1]
        bin_start = start // bin_size + 1
        bin_end = end // bin_size + 1
        mid_bin = (bin_start + bin_end) // 2
        bin_counts[mid_bin] = bin_counts.get(mid_bin, 0) + 1
    return bin_counts

def calculate_distance_counts(bin_counts, min_distance_bins=3):
    distance_counts = {}
    bins = sorted(bin_counts.keys())
    for i in range(len(bins)):
        bin1 = bins[i]
        count1 = bin_counts[bin1]
        for j in range(i, len(bins)):
            bin2 = bins[j]
            distance = bin2 - bin1
            if distance < min_distance_bins:
                continue
            count2 = bin_counts[bin2]
            if bin1 == bin2:
                pairs = count1 * (count1 - 1) // 2
            else:
                pairs = count1 * count2
            if pairs > 0:
                distance_counts[distance] = distance_counts.get(distance, 0) + pairs
    return distance_counts

def compute_pmf(distance_counts, min_prob=1e-10):
    total_pairs = sum(distance_counts.values())
    pmf = {d: c / total_pairs for d, c in distance_counts.items() if c / total_pairs >= min_prob}
    total_pmf = sum(pmf.values())
    if total_pmf > 0:
        pmf = {d: p / total_pmf for d, p in pmf.items()}
    else:
        pmf = {}
    return pmf

def calculate_symmetric_kl_divergence(pmf1, pmf2):
    all_distances = set(pmf1.keys()).union(set(pmf2.keys()))
    prob1 = np.array([pmf1.get(d, 0) for d in all_distances])
    prob2 = np.array([pmf2.get(d, 0) for d in all_distances])
    valid = (prob1 > 0) & (prob2 > 0)
    if not np.any(valid):
        return np.nan
    prob1_filtered = prob1[valid]
    prob2_filtered = prob2[valid]
    prob1_filtered /= prob1_filtered.sum()
    prob2_filtered /= prob2_filtered.sum()
    kl_div_pq = entropy(prob1_filtered, prob2_filtered)
    kl_div_qp = entropy(prob2_filtered, prob1_filtered)
    symmetric_kl = 0.5 * (kl_div_pq + kl_div_qp)
    return symmetric_kl

def main(file_path):
    data = read_data(file_path)
    chromosome_pmf = {}
    for chrom, fragments in data.items():
        bin_counts = convert_to_bin_counts(fragments)
        if len(bin_counts) < 2:
            continue
        distance_counts = calculate_distance_counts(bin_counts)
        pmf = compute_pmf(distance_counts)
        if pmf:
            chromosome_pmf[chrom] = pmf
    first_group = ["chr2L", "chr2R", "chr3L", "chr3R"]
    second_group = ["chrX", "chr4"]
    def compute_kl(chr1, chr2):
        if chr1 in chromosome_pmf and chr2 in chromosome_pmf:
            pmf1 = chromosome_pmf[chr1]
            pmf2 = chromosome_pmf[chr2]
            kl_div = calculate_symmetric_kl_divergence(pmf1, pmf2)
            print(f"KL divergence between {chr1} and {chr2}: {kl_div}")
    print("KL Divergence within group 2L, 2R, 3L, 3R:")
    for i in range(len(first_group)):
        for j in range(i+1, len(first_group)):
            compute_kl(first_group[i], first_group[j])
    print("\nKL Divergence between 2L, 2R, 3L, 3R and X, 4:")
    for chrom_main in first_group:
        for chrom_other in second_group:
            compute_kl(chrom_main, chrom_other)

if __name__ == "__main__":
    file_path = '/home/hzhou53/2024 Fall DNA and GIN model/GSM3347525NR_FDR_0.1_pseudoGEM_10000_enrichTest_master.txt'
    print("Running KL Divergence Calculation with distances ≥ 1500 bp:\n")
    main(file_path)


Running KL Divergence Calculation with distances ≥ 1500 bp:

KL Divergence within group 2L, 2R, 3L, 3R:
KL divergence between chr2L and chr2R: 0.08976398857100153
KL divergence between chr2L and chr3L: 0.056058493042677265
KL divergence between chr2L and chr3R: 0.06889050044191991
KL divergence between chr2R and chr3L: 0.11894661858189085
KL divergence between chr2R and chr3R: 0.1794493971876532
KL divergence between chr3L and chr3R: 0.10815592530212037

KL Divergence between 2L, 2R, 3L, 3R and X, 4:
KL divergence between chr2L and chrX: 0.10036556747877456
KL divergence between chr2L and chr4: 0.29302707562250435
KL divergence between chr2R and chrX: 0.05403902131894666
KL divergence between chr2R and chr4: 0.33850597931419796
KL divergence between chr3L and chrX: 0.14116686672626502
KL divergence between chr3L and chr4: 0.26853100315533873
KL divergence between chr3R and chrX: 0.20856982808513153
KL divergence between chr3R and chr4: 0.3635389479230887
